# 01 — Docker y entorno

## Proyecto Big Data

Este notebook documenta, paso a paso, la construcción del entorno reproducible del proyecto de Big Data usando Docker.

### Objetivo

Construir un entorno Linux reproducible que permita ejecutar PySpark sin depender de configuraciones locales de Windows como:

- `HADOOP_HOME`
- `winutils.exe`
- configuraciones nativas de Hadoop para Windows
- versiones locales distintas de Python o Java

La arquitectura inicial del proyecto es:

```text
Windows 11
    ↓
WSL2
    ↓
Docker Desktop
    ↓
Contenedor Linux
    ├── Python 3.11
    ├── Java 17
    └── PySpark 4.1.3
```

En esta primera etapa trabajaremos en modo local/single-node. El objetivo todavía **no** es construir un cluster distribuido.


## 1. Verificación de Docker

Primero se comprobó que el cliente de Docker estuviera instalado:

```cmd
docker --version
```

Resultado obtenido:

```text
Docker version 29.8.0, build 88096ef
```

Después se intentó ejecutar:

```cmd
docker run hello-world
```

Inicialmente apareció el error:

```text
failed to connect to the docker API at
npipe:////./pipe/dockerDesktopLinuxEngine
```

La causa fue que Docker Desktop estaba instalado, pero el motor de Docker todavía no estaba ejecutándose.

Después de iniciar Docker Desktop, el comando:

```cmd
docker run hello-world
```

se ejecutó correctamente y mostró:

```text
Hello from Docker!
This message shows that your installation appears to be working correctly.
```

### Conclusión

Con esto se validó la comunicación:

```text
Docker CLI
    ↓
Docker daemon
    ↓
Docker Desktop
    ↓
Linux containers
```


## 2. Creación del entorno base

Para mantener el entorno simple y reproducible se decidió no instalar Spark, Hadoop ni Java directamente en Windows.

Se crearon inicialmente dos archivos dentro de la carpeta `docker/`:

```text
docker/
├── Dockerfile
└── requirements.txt
```

### `requirements.txt`

```text
pyspark==4.1.3
```

Por ahora se instala únicamente PySpark. Otras dependencias como Pandas, NumPy, Jupyter o PyArrow se agregarán cuando sean necesarias.


## 3. Dockerfile inicial

El primer `Dockerfile` utilizado fue:

```dockerfile
# Imagen base con Python 3.11 sobre Linux Debian
FROM python:3.11-slim-bookworm

# Evita algunas preguntas interactivas durante instalaciones
ENV DEBIAN_FRONTEND=noninteractive

# Instalar Java 17, necesario para Spark
RUN apt-get update && \
    apt-get install -y --no-install-recommends \
        openjdk-17-jre-headless \
        procps \
    && rm -rf /var/lib/apt/lists/*

# Java utilizado por Spark
ENV JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64

# Carpeta de trabajo dentro del contenedor
WORKDIR /workspace

# Copiar dependencias Python
COPY requirements.txt .

# Instalar PySpark
RUN pip install --no-cache-dir -r requirements.txt

# Por defecto se abre Bash
CMD ["bash"]
```

### Qué hace cada parte

- `FROM python:3.11-slim-bookworm`: utiliza una imagen Linux ligera que ya contiene Python 3.11.
- `apt-get install openjdk-17-jre-headless`: instala Java 17 dentro del contenedor.
- `JAVA_HOME`: indica a Spark dónde se encuentra Java.
- `WORKDIR /workspace`: define la carpeta de trabajo del proyecto dentro del contenedor.
- `COPY requirements.txt .`: copia las dependencias Python.
- `pip install`: instala PySpark.
- `CMD ["bash"]`: permite iniciar una terminal Bash si el contenedor se ejecuta sin otro comando.


## 4. Construcción de la imagen

Desde la carpeta que contiene el `Dockerfile` se ejecutó:

```cmd
docker build -t bigdata-spark:0.1 .
```

La imagen se construyó correctamente.

El nombre elegido fue:

```text
bigdata-spark:0.1
```

donde:

- `bigdata-spark` es el nombre de la imagen.
- `0.1` es la primera versión del entorno.

La idea es evitar depender exclusivamente de la etiqueta `latest`, para mantener control sobre los cambios del entorno.


## 5. Validación del entorno

Después de construir la imagen se verificaron Python, Java y PySpark de forma independiente.

### Python

```cmd
docker run --rm bigdata-spark:0.1 python --version
```

Resultado:

```text
Python 3.11.16
```

### Java

```cmd
docker run --rm bigdata-spark:0.1 java -version
```

Resultado:

```text
openjdk version "17.0.20.1"
OpenJDK Runtime Environment
OpenJDK 64-Bit Server VM
```

### PySpark

```cmd
docker run --rm bigdata-spark:0.1 python -c "import pyspark; print(pyspark.__version__)"
```

Resultado:

```text
4.1.3
```

### Estado actual

| Componente | Estado | Versión |
|---|---|---|
| Docker Desktop | ✅ | operativo |
| Docker Engine Linux | ✅ | operativo |
| Python | ✅ | 3.11.16 |
| Java | ✅ | 17.0.20.1 |
| PySpark | ✅ | 4.1.3 |
| SparkSession | ⏳ | pendiente |
| Lectura/escritura de datos | ⏳ | pendiente |
| Volúmenes Docker | ⏳ | pendiente |
| Jupyter | ⏳ | pendiente |
| Docker Compose | ⏳ | pendiente |
| Cluster multinodo | ⏳ | futuro |


## 6. Diferencia entre imagen y contenedor

Una **imagen** es la plantilla del entorno:

```text
bigdata-spark:0.1
```

Un **contenedor** es una instancia creada a partir de esa imagen.

Por ejemplo:

```cmd
docker run --rm bigdata-spark:0.1 python --version
```

hace lo siguiente:

```text
imagen bigdata-spark:0.1
        ↓
crea un contenedor temporal
        ↓
ejecuta python --version
        ↓
muestra el resultado
        ↓
elimina el contenedor
```

La opción:

```text
--rm
```

indica que el contenedor debe eliminarse automáticamente cuando termine.

La imagen `bigdata-spark:0.1` permanece disponible.


# Próximo paso

Ahora debemos comprobar que **Spark realmente puede arrancar**, no solamente que el paquete `pyspark` está instalado.

La siguiente prueba será crear una `SparkSession` dentro del contenedor y ejecutar un DataFrame pequeño.

Esta prueba permitirá validar la cadena completa:

```text
Python
    ↓
PySpark
    ↓
Py4J
    ↓
JVM
    ↓
Java
    ↓
Spark
    ↓
DataFrame
```


## 7. Montaje del repositorio con un bind mount

Hasta este punto, Spark funcionaba dentro del contenedor, pero todavía necesitábamos conectar el sistema de archivos de Windows con el sistema de archivos del contenedor.

La raíz local del repositorio era:

```text
C:\Users\derec\Desktop\Proyectos\BigData\_project
```

Desde esa carpeta se inició un contenedor interactivo con:

```cmd
docker run --rm -it -v "%cd%:/workspace" bigdata-spark:0.1 bash
```

### Significado de las opciones

- `--rm`: elimina el contenedor cuando termina.
- `-it`: abre una terminal interactiva.
- `-v`: crea un bind mount entre una carpeta del host y una carpeta del contenedor.
- `%cd%`: en CMD representa la carpeta actual de Windows.
- `/workspace`: carpeta utilizada dentro del contenedor.

Por tanto, la relación creada fue:

```text
Windows
C:\Users\derec\Desktop\Proyectos\BigData\_project
                     │
                     │ bind mount
                     ▼
Docker / Linux
/workspace
```

Dentro del contenedor se verificó:

```bash
pwd
```

Resultado:

```text
/workspace
```

Después:

```bash
ls
```

Resultado:

```text
DECISIONS.md
PROJECT_STATE.md
README.md
ROADMAP.md
data
docker
docs
notebooks
sources
src
tests
```

### Conclusión

El contenedor puede acceder directamente al repositorio real de Windows.

Esto permite modificar código en VS Code y ejecutar los cambios inmediatamente dentro de Docker, sin reconstruir la imagen cada vez.


## 8. Smoke test reproducible de Spark

En lugar de seguir usando un comando largo con `python -c`, se creó una prueba reproducible:

```text
tests/
└── smoke_test_spark.py
```

El script crea una `SparkSession`, consulta las versiones principales y genera un pequeño DataFrame.

Código base:

```python
from pyspark.sql import SparkSession


def main():
    spark = (
        SparkSession.builder
        .master("local[*]")
        .appName("bigdata-smoke-test")
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("WARN")

    print("\n" + "=" * 60)
    print("PRUEBA DEL ENTORNO SPARK")
    print("=" * 60)

    print(f"Spark version: {spark.version}")

    java_version = (
        spark.sparkContext
        ._jvm.java.lang.System
        .getProperty("java.version")
    )

    print(f"Java version: {java_version}")

    hadoop_version = (
        spark.sparkContext
        ._jvm.org.apache.hadoop.util.VersionInfo
        .getVersion()
    )

    print(f"Hadoop version: {hadoop_version}")

    print("\nDataFrame de prueba:")

    df = spark.range(5)

    df.show()

    assert df.count() == 5

    spark.stop()


if __name__ == "__main__":
    main()
```

La prueba se ejecutó dentro del contenedor con:

```bash
python tests/smoke_test_spark.py
```

### Resultado observado

```text
Spark version: 4.1.3
Java version: 17.0.20.1
Hadoop version: 3.4.2
```

y Spark mostró correctamente:

```text
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+
```

### Qué validó esta prueba

```text
Python
  ↓
PySpark
  ↓
Py4J
  ↓
JVM
  ↓
Java
  ↓
Spark
  ↓
Hadoop
  ↓
DataFrame
```

Esto confirmó que Spark no solamente estaba instalado: el motor podía iniciarse y ejecutar operaciones reales.


## 9. Prueba de persistencia con Parquet

El siguiente objetivo fue comprobar que Spark pudiera escribir datos desde el contenedor y que esos datos persistieran físicamente en el repositorio de Windows.

Se amplió `tests/smoke_test_spark.py` con:

```python
print("\n" + "=" * 60)
print("PRUEBA DE ESCRITURA Y LECTURA")
print("=" * 60)

output_path = "/workspace/data/_smoke_test/parquet"

df.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"Datos escritos en: {output_path}")

df_read = spark.read.parquet(output_path)

print("\nDatos leídos nuevamente:")

df_read.show()

assert df_read.count() == 5

print("Escritura y lectura Parquet correctas.")
```

El script volvió a ejecutarse:

```bash
python tests/smoke_test_spark.py
```

Resultado relevante:

```text
Datos escritos en: /workspace/data/_smoke_test/parquet
```

Después Spark pudo leer nuevamente los archivos generados:

```text
+---+
| id|
+---+
|  0|
|  1|
|  3|
|  4|
|  2|
+---+
```

El orden distinto de las filas no representa un problema. En Spark los DataFrames son distribuidos y no garantizan un orden de filas salvo que se aplique explícitamente un `orderBy()`.

### Qué se validó

```text
Código en Windows
      ↓
bind mount
      ↓
contenedor Linux
      ↓
Spark
      ↓
DataFrame
      ↓
Parquet
      ↓
/workspace/data/
      ↓
bind mount
      ↓
data/ en Windows
```

Con esta prueba se validó tanto la ejecución de Spark como la persistencia de archivos fuera del ciclo de vida del contenedor.


## 10. Estado del entorno después de las pruebas

| Componente | Estado | Resultado |
|---|---|---|
| Docker Desktop | ✅ | operativo |
| Docker Engine Linux | ✅ | operativo |
| Imagen `bigdata-spark:0.1` | ✅ | construida |
| Python | ✅ | 3.11.16 |
| Java | ✅ | 17.0.20.1 |
| PySpark / Spark | ✅ | 4.1.3 |
| Hadoop utilizado por Spark | ✅ | 3.4.2 |
| SparkSession | ✅ | funcional |
| DataFrame | ✅ | funcional |
| Bind mount del repositorio | ✅ | funcional |
| Escritura Parquet | ✅ | funcional |
| Lectura Parquet | ✅ | funcional |
| Persistencia host ↔ contenedor | ✅ | funcional |
| Ejecución simplificada | ⏳ | siguiente paso |
| Docker Compose | ⏳ | pendiente |
| Jupyter dentro del entorno | ⏳ | pendiente |
| Cluster multinodo | ⏳ | futuro |

### Arquitectura actual

```text
Windows 11
    │
    ├── VS Code
    └── repositorio del proyecto
            │
            │ bind mount
            ▼
        Docker Desktop
            │
            ▼
        Contenedor Linux
            ├── Python 3.11
            ├── Java 17
            ├── Spark 4.1.3
            └── Hadoop 3.4.2
```

### Nota sobre el warning de Hadoop

Durante el inicio aparece:

```text
WARN NativeCodeLoader:
Unable to load native-hadoop library for your platform...
using builtin-java classes where applicable
```

Para el entorno local actual no impide la ejecución de Spark. Las pruebas de creación de DataFrames y de lectura/escritura Parquet terminaron correctamente.


## 11. Próximo paso

Actualmente el flujo manual requiere:

```text
1. Abrir Docker Desktop
2. Ir a la raíz del repositorio
3. Ejecutar docker run con el bind mount
4. Entrar a Bash
5. Ejecutar el script Python
```

El siguiente objetivo será simplificar este flujo para ejecutar comandos del proyecto directamente desde Windows, sin tener que entrar manualmente al contenedor.

Después de eso podremos evaluar si conviene introducir `docker compose` para centralizar:

- imagen utilizada;
- carpeta de trabajo;
- bind mounts;
- variables de entorno;
- comandos;
- configuración futura de Spark.



## 12. Docker Compose

Después de validar que el entorno funcionaba con `docker run`, se agregó Docker Compose para centralizar la configuración.

Archivo creado en la raíz del proyecto:

```text
_project/
├── compose.yaml
├── docker/
│   ├── Dockerfile
│   └── requirements.txt
├── data/
├── docs/
├── notebooks/
├── src/
└── tests/
```

Contenido actual de `compose.yaml`:

```yaml
services:
  spark:
    build:
      context: ./docker
      dockerfile: Dockerfile

    image: bigdata-spark:0.1

    working_dir: /workspace

    volumes:
      - .:/workspace

    stdin_open: true
    tty: true
```

### Qué significa cada parte

- `services`: define los servicios administrados por Compose.
- `spark`: nombre de nuestro servicio.
- `build.context`: indica dónde está el contexto de construcción.
- `dockerfile`: especifica el Dockerfile a utilizar.
- `image`: asigna el nombre `bigdata-spark:0.1`.
- `working_dir`: fija `/workspace` como directorio de trabajo.
- `volumes`: monta la raíz del repositorio en `/workspace`.
- `stdin_open` y `tty`: permiten abrir una terminal interactiva.

Una forma útil de recordarlo:

```text
Dockerfile     = cómo construir el entorno
requirements   = qué paquetes Python instalar
compose.yaml   = cómo ejecutar el entorno
```



# Referencia de comandos Docker

Esta sección funciona como manual rápido de los comandos utilizados en el proyecto.

## 13. Verificar Docker

```cmd
docker --version
```

Comprueba que el cliente Docker está instalado.

---

## 14. Probar el Docker Engine

```cmd
docker run hello-world
```

Si aparece:

```text
Hello from Docker!
```

Docker puede comunicarse con el daemon y ejecutar contenedores Linux.

---

## 15. Construir una imagen

```cmd
docker build -t bigdata-spark:0.1 .
```

Desglose:

```text
docker build             construye una imagen
-t                       asigna nombre y tag
bigdata-spark            nombre
0.1                      versión/tag
.                        build context actual
```

---

## 16. Ver imágenes locales

```cmd
docker images
```

Permite comprobar que `bigdata-spark:0.1` existe.

---

## 17. Ejecutar un comando temporal

```cmd
docker run --rm bigdata-spark:0.1 python --version
```

`--rm` elimina el contenedor al terminar, pero conserva la imagen.

Otros ejemplos usados:

```cmd
docker run --rm bigdata-spark:0.1 java -version
```

```cmd
docker run --rm bigdata-spark:0.1 python -c "import pyspark; print(pyspark.__version__)"
```

---

## 18. Abrir una terminal interactiva

```cmd
docker run --rm -it bigdata-spark:0.1 bash
```

- `-i`: mantiene la entrada abierta.
- `-t`: crea una terminal virtual.

---

## 19. Montar el repositorio

Desde la raíz del proyecto:

```cmd
docker run --rm -it -v "%cd%:/workspace" bigdata-spark:0.1 bash
```

La opción:

```text
-v HOST:CONTENEDOR
```

crea un bind mount.

En nuestro caso:

```text
Windows
C:\...\_project
       │
       ▼
Docker
/workspace
```

---

## 20. Ejecutar un script sin entrar a Bash

```cmd
docker run --rm -v "%cd%:/workspace" bigdata-spark:0.1 python tests/smoke_test_spark.py
```

Este comando crea un contenedor temporal, monta el repositorio, ejecuta Spark y elimina el contenedor al finalizar.

---

## 21. Validar Compose

```cmd
docker compose config
```

Sirve para comprobar que `compose.yaml` tiene una sintaxis válida y ver cómo Docker interpreta la configuración.

---

## 22. Construir con Compose

```cmd
docker compose build
```

Compose utiliza la sección:

```yaml
build:
  context: ./docker
  dockerfile: Dockerfile
```

para construir la imagen.

Si una capa no cambió puede aparecer:

```text
CACHED
```

Esto significa que Docker reutilizó una capa anterior.

---

## 23. Ejecutar Spark con Compose

```cmd
docker compose run --rm spark python tests/smoke_test_spark.py
```

Desglose:

```text
docker compose run
        │
        ├── --rm
        ├── spark     servicio
        └── python tests/smoke_test_spark.py
```

Este es el comando recomendado actualmente para validar el entorno.

---

## 24. Abrir Bash con Compose

```cmd
docker compose run --rm spark bash
```

Una vez dentro pueden ejecutarse:

```bash
pwd
ls
python --version
java -version
```

---

## 25. Ver contenedores

Solo activos:

```cmd
docker ps
```

Todos:

```cmd
docker ps -a
```

---

## 26. Revisar espacio utilizado

```cmd
docker system df
```

Muestra el espacio ocupado por imágenes, contenedores y caché.

---

## 27. Limpiar recursos no utilizados

```cmd
docker system prune
```

Debe utilizarse con cuidado porque elimina recursos Docker no utilizados.

---

## 28. Eliminar una imagen concreta

```cmd
docker rmi bigdata-spark:0.1
```

La imagen puede reconstruirse posteriormente con:

```cmd
docker compose build
```



## 29. Archivos Docker actuales del proyecto

### `docker/Dockerfile`

```dockerfile
FROM python:3.11-slim-bookworm

ENV DEBIAN_FRONTEND=noninteractive

RUN apt-get update && \
    apt-get install -y --no-install-recommends \
        openjdk-17-jre-headless \
        procps \
    && rm -rf /var/lib/apt/lists/*

ENV JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64

WORKDIR /workspace

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

CMD ["bash"]
```

### `docker/requirements.txt`

```text
pyspark==4.1.3
```

### `compose.yaml`

```yaml
services:
  spark:
    build:
      context: ./docker
      dockerfile: Dockerfile

    image: bigdata-spark:0.1

    working_dir: /workspace

    volumes:
      - .:/workspace

    stdin_open: true
    tty: true
```

### Responsabilidad de cada archivo

```text
Dockerfile
    → construye el entorno Linux

requirements.txt
    → define dependencias Python

compose.yaml
    → define cómo se ejecuta el entorno
```



## 30. Flujo recomendado de trabajo

### Si solo cambia el código

Por ejemplo:

```text
src/
tests/
notebooks/
```

No es necesario reconstruir la imagen.

Ejecutar directamente:

```cmd
docker compose run --rm spark python tests/smoke_test_spark.py
```

Los cambios ya están visibles dentro del contenedor gracias al bind mount.

---

### Si cambia `requirements.txt`

Ejecutar:

```cmd
docker compose build
```

y después:

```cmd
docker compose run --rm spark ...
```

---

### Si cambia el `Dockerfile`

También es necesario:

```cmd
docker compose build
```

Ejemplos:

- cambiar Python;
- cambiar Java;
- instalar paquetes Linux;
- modificar dependencias base.

---

### Regla práctica

```text
¿Cambió Dockerfile?
        │
        ├── Sí → docker compose build
        │
        └── No
             │
¿Cambió requirements.txt?
        │
        ├── Sí → docker compose build
        │
        └── No
             │
             ▼
ejecutar directamente
docker compose run ...
```



## 31. Arquitectura de la versión 0.1

```text
Windows 11
│
├── VS Code
├── Docker Desktop
├── WSL2
│
└── _project/
    ├── compose.yaml
    ├── docker/
    ├── data/
    ├── notebooks/
    ├── src/
    └── tests/
         │
         │ bind mount
         ▼
Docker container
└── /workspace
    ├── Python 3.11.16
    ├── Java 17.0.20.1
    ├── Spark / PySpark 4.1.3
    └── Hadoop 3.4.2
```

### Estado validado

```text
Docker            ✅
Python            ✅
Java              ✅
PySpark           ✅
SparkSession      ✅
Hadoop            ✅
DataFrame         ✅
Parquet           ✅
Bind mounts       ✅
Docker Compose    ✅
```

Esta es nuestra primera versión funcional y reproducible del entorno local.
